In [ ]:
!pip install -q transformers sentencepiece pandas openpyxl tqdm

In [ ]:
import pandas as pd
from tqdm import tqdm
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
import torch

In [ ]:
MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
model = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print("Device:", device)

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Device: cpu


In [ ]:
INPUT_FILE = "hats_dataset.xlsx"
TEXT_COL = "Formatted Question"

df = pd.read_excel(INPUT_FILE)
print("Rows loaded:", len(df))

df["Telugu"] = ""
df["Hindi_back"] = ""

Rows loaded: 405


In [ ]:
def translate(text, source_lang_code, target_lang_code, max_length=128):
    """
    Translate using MBart-50
    source_lang_code: 'hi_IN' or 'te_IN'
    target_lang_code: 'te_IN' or 'hi_IN'
    """
    tokenizer.src_lang = source_lang_code
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.lang_code_to_id[target_lang_code],
            max_length=max_length
        )
    translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translated_text

In [ ]:
for i, text in enumerate(tqdm(df[TEXT_COL].astype(str).tolist(), desc="Hindi -> Telugu")):
    try:
        telugu_text = translate(text, source_lang_code="hi_IN", target_lang_code="te_IN")
        df.loc[i, "Telugu"] = telugu_text
    except Exception as e:
        df.loc[i, "Telugu"] = f"[ERROR] {e}"

Hindi -> Telugu: 100%|██████████| 405/405 [28:35<00:00,  4.24s/it]


In [ ]:
for i, text in enumerate(tqdm(df["Telugu"].astype(str).tolist(), desc="Telugu -> Hindi")):
    try:
        hindi_back_text = translate(text, source_lang_code="te_IN", target_lang_code="hi_IN")
        df.loc[i, "Hindi_back"] = hindi_back_text
    except Exception as e:
        df.loc[i, "Hindi_back"] = f"[ERROR] {e}"

Telugu -> Hindi: 100%|██████████| 405/405 [16:20<00:00,  2.42s/it]


In [ ]:
OUTPUT_FILE = "translated_output_mbart50.xlsx"
df.to_excel(OUTPUT_FILE, index=False)
print("Saved:", OUTPUT_FILE)

Saved: translated_output_mbart50.xlsx
